# ShopEase Demo 3: Function Routing with Groq

The model selects one support action from a list of tools. Notebook code runs a mock backend action, returns its result to the model, and asks for a customer-facing reply. This is a small tool-use loop using Groq directly; OpenRouter and Claude are not used.

For live calls, set `GROQ_API_KEY` before launching Jupyter or add it to a local `Module_3/.env` copied from `.env.example`. Without it, an explicitly labeled offline example demonstrates the same routing and mock backend flow.

In [1]:
import os
from pathlib import Path
from groq import Groq

def find_groq_key():
    if os.getenv("GROQ_API_KEY"):
        return os.environ["GROQ_API_KEY"]
    # Optionally read a local Module_3/.env file; the notebook never stores the key.
    for folder in (Path.cwd(), *Path.cwd().parents):
        env_file = folder / ".env"
        if env_file.is_file():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if line.startswith("GROQ_API_KEY="):
                    value = line.split("=", 1)[1].strip().strip("\"'")
                    if value and value != "your_groq_key_here":
                        return value
    return None

MODEL = "openai/gpt-oss-120b"
GROQ_API_KEY = find_groq_key()
LIVE_API = bool(GROQ_API_KEY)
client = Groq(api_key=GROQ_API_KEY) if LIVE_API else None
print("Mode:", "LIVE Groq API" if LIVE_API else "OFFLINE PREVIEW (no API request)")

Mode: LIVE Groq API


## 1. Define tools and a mock backend

The model only selects a tool and supplies arguments. The Python application executes the tool. These backend functions return example data; they do not connect to a real order or refund system.

In [2]:
import json

TOOLS = [
    {"type": "function", "function": {
        "name": "check_order_status", "description": "Look up shipping status for an order.",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}
    }},
    {"type": "function", "function": {
        "name": "initiate_refund", "description": "Start a mock refund for an order.",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}, "reason": {"type": "string"}}, "required": ["order_id", "reason"]}
    }},
    {"type": "function", "function": {
        "name": "track_shipment", "description": "Get tracking details for an order.",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}
    }},
    {"type": "function", "function": {
        "name": "escalate_to_human", "description": "Escalate a complex or upset customer case.",
        "parameters": {"type": "object", "properties": {"reason": {"type": "string"}}, "required": ["reason"]}
    }},
]

MOCK_BACKEND = {
    "check_order_status": lambda order_id: {"order_id": order_id, "status": "out for delivery", "eta": "today by 8pm"},
    "initiate_refund": lambda order_id, reason: {"order_id": order_id, "refund_status": "initiated", "reason_logged": reason},
    "track_shipment": lambda order_id: {"order_id": order_id, "carrier": "BlueDart", "last_scan": "regional hub", "eta": "tomorrow"},
    "escalate_to_human": lambda reason: {"escalated": True, "priority": "high", "reason_logged": reason},
}

def run_tool(name, arguments):
    return MOCK_BACKEND[name](**arguments)

print("Available tools:", [t["function"]["name"] for t in TOOLS])

Available tools: ['check_order_status', 'initiate_refund', 'track_shipment', 'escalate_to_human']


## 2. Route, act, observe, respond

In [3]:
CUSTOMER_MESSAGE = "Hey, where's my order SE-71234? It was supposed to come yesterday."

def route_message(text):
    messages = [{"role": "user", "content": text}]
    if LIVE_API:
        first = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0,
            max_tokens=400,
        )
        assistant_message = first.choices[0].message
        if not assistant_message.tool_calls:
            return {"tool": None, "reply": assistant_message.content}
        call = assistant_message.tool_calls[0]
        name = call.function.name
        arguments = json.loads(call.function.arguments)
        messages.append({"role": "assistant", "content": None, "tool_calls": [call.model_dump(exclude_none=True)]})
    else:
        # Offline preview route for the sample order-status request.
        name, arguments = "check_order_status", {"order_id": "SE-71234"}

    backend_result = run_tool(name, arguments)
    if LIVE_API:
        messages.append({"role": "tool", "tool_call_id": call.id, "name": name, "content": json.dumps(backend_result)})
        final = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, temperature=0.2, max_tokens=250)
        reply = final.choices[0].message.content
    else:
        reply = f"Your order {arguments.get('order_id')} is out for delivery and is expected today by 8pm. (offline example)"
    return {"tool": name, "arguments": arguments, "mock_backend_result": backend_result, "customer_reply": reply}

outcome = route_message(CUSTOMER_MESSAGE)
print(json.dumps(outcome, indent=2))

{
  "tool": "check_order_status",
  "arguments": {
    "order_id": "SE-71234"
  },
  "mock_backend_result": {
    "order_id": "SE-71234",
    "status": "out for delivery",
    "eta": "today by 8pm"
  },
  "customer_reply": "I\u2019m sorry you didn\u2019t receive your package when expected. According to our system, order **SE\u201171234** is currently **out for delivery** and is scheduled to arrive **today by 8\u202fPM**.\n\nIf you\u2019d like a real\u2011time tracking link or have any other concerns, just let me know\u2014I\u2019m happy to help!"
}


## Discussion

Trace the sequence: user message → tool choice → mock backend execution → tool result → final response. The model does not directly execute Python or access real customer records. Change the sample message to a refund or escalation request and observe the routing in live mode.